# County Covariates from HDPulse

This notebook ingests the three HDPulse county exports you downloaded and builds a merged county covariate table for heterogeneity analysis.

Inputs:
- `data/raw/hdpulse/county_income_median_hh_2019_2023.csv`
- `data/raw/hdpulse/county_poverty_families_2019_2023.csv`
- `data/raw/hdpulse/county_unemployment_2019_2023.csv`

Output:
- `data/processed/county_hdpulse_covariates_2019_2023.csv`


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path('/Users/jomus/Code/capstone')
RAW = ROOT / "data" / "raw" / "hdpulse"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

RAW, PROCESSED


(PosixPath('/Users/jomus/Code/capstone/data/raw/hdpulse'),
 PosixPath('/Users/jomus/Code/capstone/data/processed'))

In [2]:
def read_hdpulse(path: Path, value_col_name: str, extra_cols: list[str] | None = None) -> pd.DataFrame:
    # HDPulse exports include a 4-line preamble before the CSV header.
    df = pd.read_csv(path, skiprows=4, encoding="utf-8-sig", dtype={"FIPS": "string"})

    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={"Value (Dollars)": value_col_name, "Value (Percent)": value_col_name})

    keep = ["County", "FIPS", value_col_name]
    if extra_cols:
        for c in extra_cols:
            if c in df.columns:
                keep.append(c)

    df = df[keep].copy()
    df["County"] = df["County"].astype(str).str.strip()
    df["FIPS"] = df["FIPS"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(5)
    # HDPulse adds citation/notes rows at the end; keep only true county FIPS rows.
    df = df[df["FIPS"].str.fullmatch(r"\d{5}", na=False)].copy()

    # Remove commas/percent symbols and coerce numeric values.
    df[value_col_name] = (
        df[value_col_name]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
    )
    df[value_col_name] = pd.to_numeric(df[value_col_name], errors="coerce")

    return df


In [3]:
income = read_hdpulse(
    RAW / "county_income_median_hh_2019_2023.csv",
    value_col_name="median_household_income_usd",
)

poverty = read_hdpulse(
    RAW / "county_poverty_families_2019_2023.csv",
    value_col_name="family_poverty_rate_pct",
    extra_cols=["Families (Below Poverty)"],
)

unemp = read_hdpulse(
    RAW / "county_unemployment_2019_2023.csv",
    value_col_name="unemployment_rate_pct",
    extra_cols=["People (Unemployed)"],
)

income.head(), poverty.head(), unemp.head()


(                           County   FIPS  median_household_income_usd
 0                     Puerto Rico  72001                      25096.0
 1        Randolph County, Georgia  13243                      25425.0
 2    Jackson County, South Dakota  46071                      26686.0
 3  East Carroll Parish, Louisiana  22035                      28321.0
 4          Presidio County, Texas  48377                      29014.0,
                                County   FIPS  family_poverty_rate_pct  \
 0  Oglala Lakota County, South Dakota  46102                     48.8   
 1        Jackson County, South Dakota  46071                     38.9   
 2           Todd County, South Dakota  46121                     38.4   
 3                         Puerto Rico  72001                     38.2   
 4                Dimmit County, Texas  48127                     36.8   
 
    Families (Below Poverty)  
 0                    1101.0  
 1                     213.0  
 2                     672.0  
 3 

In [4]:
county_covars = (
    income.merge(
        poverty.drop(columns=["County"], errors="ignore"),
        on=["FIPS"],
        how="outer",
    )
    .merge(
        unemp.drop(columns=["County"], errors="ignore"),
        on=["FIPS"],
        how="outer",
    )
)

# Keep canonical county name from income; fill from other files where missing.
county_covars = county_covars.rename(columns={"County": "county_name"})
if "county_name" not in county_covars.columns:
    county_covars["county_name"] = pd.NA

# Keep a simple vulnerability score for optional heterogeneity bins.
z_income = (county_covars["median_household_income_usd"] - county_covars["median_household_income_usd"].mean()) / county_covars["median_household_income_usd"].std(ddof=0)
z_poverty = (county_covars["family_poverty_rate_pct"] - county_covars["family_poverty_rate_pct"].mean()) / county_covars["family_poverty_rate_pct"].std(ddof=0)
z_unemp = (county_covars["unemployment_rate_pct"] - county_covars["unemployment_rate_pct"].mean()) / county_covars["unemployment_rate_pct"].std(ddof=0)

county_covars["economic_vulnerability_z"] = (-z_income + z_poverty + z_unemp) / 3

county_covars = county_covars.sort_values("FIPS").reset_index(drop=True)
county_covars.head()


,county_name,FIPS,median_household_income_usd,family_poverty_rate_pct,Families (Below Poverty),unemployment_rate_pct,People (Unemployed),economic_vulnerability_z
0,"Autauga County, Alabama",01001,69841.0,8.1,1276.0,2.5,688.0,-0.511323
1,"Baldwin County, Alabama",01003,75019.0,7.3,4742.0,3.2,3615.0,-0.565013
2,"Barbour County, Alabama",01005,44290.0,17.7,1058.0,5.7,518.0,1.032572
3,"Bibb County, Alabama",01007,51215.0,16.0,843.0,10.0,936.0,1.384747
4,"Blount County, Alabama",01009,61096.0,10.3,1607.0,5.8,1586.0,0.252049


In [5]:
out_path = PROCESSED / "county_hdpulse_covariates_2019_2023.csv"
county_covars.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Rows: {len(county_covars):,}")
print(f"Unique FIPS: {county_covars['FIPS'].nunique():,}")


Saved: /Users/jomus/Code/capstone/data/processed/county_hdpulse_covariates_2019_2023.csv
Rows: 3,143
Unique FIPS: 3,143


In [6]:
# Quick null profile for merge diagnostics
county_covars[[
    "median_household_income_usd",
    "family_poverty_rate_pct",
    "unemployment_rate_pct",
    "economic_vulnerability_z",
]].isna().mean().sort_values(ascending=False)


median_household_income_usd    0.000636
economic_vulnerability_z       0.000636
family_poverty_rate_pct        0.000000
unemployment_rate_pct          0.000000
dtype: float64